# 월간 보고서 작성

- 분류별 지출 항목
  - 분류별 지출 표 / 그래프
- 전체 수입 총계
- 수입 - 지출, 월별 수입 총계
- 그래프를 md5 형태로 출력

## 0. 초기화

In [1]:

#!uv pip install plotly
#!uv pip install pypandoc
#!uv pip install nbformat
#!uv pip install --upgrade kaleido

In [2]:
import pandas as pd
import sqlite3
import ledgerly
from ledgerly.utils import find_project_root


## 1. 지출 데이터 불러오기

In [3]:

conn = sqlite3.connect(find_project_root() /"data" /"db"/"ledgerly.db")
def get_expense_summary(start_date: str, end_date: str) -> pd.DataFrame:

    expense_df = pd.read_sql_query(f"""
    SELECT
        category,
        merchant_name,
        SUM(amount) AS total_amount
    FROM expenditure
    WHERE used_at >= '{start_date}'
    AND used_at < '{end_date}'
    GROUP BY category, merchant_name
    ORDER BY total_amount DESC
    """, conn)

    return expense_df

In [4]:

expense_summary_df = get_expense_summary("2026-01-01", "2026-02-01")
expense_summary_df

,category,merchant_name,total_amount
0,대출이자,주담대 이자,1357325.0
1,쇼핑,네이버페이,660110.0
2,저축,맑은하늘적금,200000.0
3,식비,홈플러스㈜,189390.0
4,쇼핑,쿠팡(쿠페이),183630.0
5,용돈,진영 용돈,150000.0
6,보험,진영 보험,146000.0
7,기타,NICE 결제대행_4,145000.0
8,용돈,준열 용돈,130000.0
9,기타,(주)딜리유통,114000.0


## 2. 수입 데이터 불러오기

In [5]:
def get_income_summary(start_date: str, end_date: str) -> pd.DataFrame:
    income_df = pd.read_sql_query(f"""
   SELECT SUM(amount) AS total_income, income_type
    FROM income
    WHERE income_date >= '{start_date}'
    AND income_date < '{end_date}'
    GROUP BY income_type
    """, conn)



    return income_df


income_summary_df = get_income_summary("2026-01-01", "2026-02-01")
income_summary_df

,total_income,income_type
0,400000,과외비
1,42502,배당
2,1000000,부모급여
3,100000,아동수당


## 3. 총 수입 데이터 markdown 생성

In [6]:
TOTAL_INCOME_MARKDOWN_TEMPLATE = """
# 2026년 1월 월간 보고서

## 1. 순수익 요약

| 항목 | 금액 (원) |
|------|-----------|
| 총 수입 | {total_income:,} |
| 총 지출 | {total_expense:,} |
| 총 지출 (저축/투자 제외) | {expense_excluding_invest:,} |
| 순수익 (수입 - 지출) | {net_profit:,} |
| 실질 순수익 (저축/투자 제외) | {adjusted_net:,} |
"""

In [7]:
total_expense = expense_summary_df.sum(numeric_only=True)["total_amount"]
total_income = income_summary_df.sum(numeric_only=True)["total_income"]
expense_excluding_invest = expense_summary_df[~expense_summary_df["category"].isin(["저축", "투자"])].sum(numeric_only=True)["total_amount"]
net_profit = total_income - total_expense
adjusted_net = total_income - expense_excluding_invest

In [8]:
TOTAL_INCOME_MARKDOWN_TEMPLATE.format(
    total_income=total_income,
    total_expense=total_expense,
    expense_excluding_invest=expense_excluding_invest,
    net_profit=net_profit,
    adjusted_net=adjusted_net
)

'\n# 2026년 1월 월간 보고서\n\n## 1. 순수익 요약\n\n| 항목 | 금액 (원) |\n|------|-----------|\n| 총 수입 | 1,542,502 |\n| 총 지출 | 4,996,320.0 |\n| 총 지출 (저축/투자 제외) | 4,576,320.0 |\n| 순수익 (수입 - 지출) | -3,453,818.0 |\n| 실질 순수익 (저축/투자 제외) | -3,033,818.0 |\n'

In [9]:
# with open("2026-01-report.md", "w", encoding="utf-8") as f:
#     f.write(markdown_text)


## 4. 수입 상세 보고서

In [10]:
INCOME_DETAIL_TEMPLATE = """
## 2. 수입 상세

| 수입 형태 | 금액 (원) |
|------------|-----------|
{income_rows}
"""

In [11]:
income_rows = ""
for _, row in income_summary_df.iterrows():
    income_rows += f"| {row['income_type']} | {row['total_income']:,} |\n"
income_rows

'| 과외비 | 400,000 |\n| 배당 | 42,502 |\n| 부모급여 | 1,000,000 |\n| 아동수당 | 100,000 |\n'

## 5. 지출 상세 보고서

In [12]:

EXPENSE_CATEGORY_TEMPLATE = """
## 3-1. 지출 분류별 요약

| 분류 | 금액 (원) |
|----------|-----------|
{expense_rows}
"""

EXPENSE_CATEGORY_DETAIL_TEMPLATE = """
## 3-3. 지출 분류별 상세
| 분류 |사용처| 금액 (원) |
|----------|-----------|-----------|
{expense_detail_rows}
"""

EXPENSE_CHART_TEMPLATE = """
## 3-2. 지출 분류 차트
![지출 분류 차트 보기]({chart_file_path})
"""

In [13]:
category_summary = (
    expense_summary_df
    .groupby("category", as_index=False)["total_amount"]
    .sum()
    .sort_values(by="total_amount", ascending=False)

)
category_summary

,category,total_amount
3,대출이자,1457325.0
7,쇼핑,913540.0
2,기타,525800.0
5,보험,427921.0
12,저축,420000.0
10,용돈,280000.0
8,식비,276880.0
0,경조사,200000.0
1,교통,167100.0
13,통신,132580.0


In [14]:
total_sum = category_summary["total_amount"].sum()

In [15]:
expense_rows = ""
for _, row in category_summary.iterrows():
    expense_rows += f"| {row['category']} | {int(row['total_amount']):,} |\n"

expense_rows += f"| **총 지출** | **{total_sum:,}** |\n"


In [16]:
import plotly.express as px
fig = px.pie(
    category_summary,
    names = "category",
    values = "total_amount",
    title="2026년 1월 지출 분류별 비율"
)
fig.update_traces(textinfo = "percent+label")


In [17]:

chart_file_path = "2026-01-expense-category-chart.png"
# fig.write_html(chart_file_path)
fig.write_image(chart_file_path)

In [18]:
detail_rows = ""

for _, row in expense_summary_df.iterrows():
    detail_rows += f"| {row['category']} | {row['merchant_name']} | {int(row['total_amount']):,} |\n"

detail_total = expense_summary_df["total_amount"].sum()

detail_rows += f"| **총합** |  | **{int(detail_total):,}** |\n"

In [19]:
markdown_txt = (
    TOTAL_INCOME_MARKDOWN_TEMPLATE.format(
        total_income=total_income,
        total_expense=total_expense,
        expense_excluding_invest=expense_excluding_invest,
        net_profit=net_profit,
        adjusted_net=adjusted_net
    )
    + INCOME_DETAIL_TEMPLATE.format(income_rows=income_rows)
    + EXPENSE_CATEGORY_TEMPLATE.format(expense_rows=expense_rows)
    + EXPENSE_CHART_TEMPLATE.format(chart_file_path=chart_file_path)
    + EXPENSE_CATEGORY_DETAIL_TEMPLATE.format(expense_detail_rows=detail_rows)
)

In [20]:

with open("2026-01-report.md", "w", encoding="utf-8") as f:
    f.write(markdown_txt)

In [21]:
#!uv pip install pypandoc

In [24]:
import pypandoc

output_file = "2026-01-report.pdf"

pypandoc.convert_text(
    markdown_txt,
    to="pdf",
    format="md",
    outputfile=output_file,
    # extra_args=["--pdf-engine=xelatex"]
    extra_args=["--pdf-engine=wkhtmltopdf"]

)

[WARNING] Loading pages (1/6)
Counting pages (2/6)                                               
Resolving links (4/6)                                                       
Loading headers and footers (5/6)                                           
Printing pages (6/6)
Done                                                                      



''